# Modelling 2: Significance, robustness and interpretation
**Input:** the models and predictions from notebook 1. This notebook asks whether gated fusion really differs from early fusion, what
happens when a modality is missing, and what the gate learned.

In [ ]:
# Make the shared code in src/ importable, wherever the repository folder is
import os, sys
root = os.getcwd()
while not os.path.isdir(os.path.join(root, "src", "mvsa")) and root != os.path.dirname(root):
    root = os.path.dirname(root)
sys.path.insert(0, os.path.join(root, "src"))

from mvsa.modeling import *

state = load_state("run_state")
PRED, GATES, best_fam, best_setup, CFG = (state[k] for k in ("PRED", "GATES", "best_fam", "best_setup", "CFG"))

## B6. Is gated fusion significantly different from early fusion? (McNemar, seed 0)
McNemar's test only looks at posts where the two models *disagree*: `b` = early right and gated wrong,
`c` = gated right and early wrong. If the models were equally good, `b` and `c` would be about equal.

In [10]:
yte = D["test"]["y"]
for fam in FAMILIES:
    # b and c count the posts that only one of the two models got right
    e_ok, g_ok = PRED[(fam, "early", 0)] == yte, PRED[(fam, "gated", 0)] == yte
    b, c = int((e_ok & ~g_ok).sum()), int((~e_ok & g_ok).sum())
    p = binomtest(b, b + c, 0.5).pvalue if b + c else 1.0     # exact form of McNemar's test
    print(f"{fam:14s} early-only correct={b:3d}  gated-only correct={c:3d}  p={p:.3f}  "
          f"{'significant' if p < 0.05 else 'not significant'} at 5%")

logreg         early-only correct= 38  gated-only correct= 19  p=0.016  significant at 5%
linear_svm     early-only correct= 57  gated-only correct= 31  p=0.007  significant at 5%
random_forest  early-only correct= 78  gated-only correct= 60  p=0.148  not significant at 5%
mlp            early-only correct= 45  gated-only correct= 27  p=0.044  significant at 5%


## B7. Robustness: what if one modality is missing?
The image (or text) embedding of every test post is replaced by the **training-set mean** embedding, so the model
receives an average input for that modality.
**Read macro-F1, not just accuracy.** The mean embedding isn't truly neutral: the unimodal models predict "positive"
for every post when given it, so masking pushes predictions towards the majority class and can flatter accuracy.
`pos_*` shows the share of posts predicted positive, which makes this visible.

In [11]:
def masked(d, modality):
    """Copy a split with one modality replaced by its training-set mean embedding, to simulate it being missing."""
    m = dict(d)
    m[modality] = np.repeat(D["train"][modality].mean(0, keepdims=True), len(d["ids"]), axis=0)
    return m

predictors, _ = build(best_fam, CFG[best_fam], 0)
rob = []
for setup in ["early", "late", "gated"]:
    row = {"setup": setup}
    for name, d in [("full", D["test"]), ("no_image", masked(D["test"], "image")), ("no_text", masked(D["test"], "text"))]:
        P = predictors[setup](d)
        row[f"acc_{name}"] = accuracy_score(D["test"]["y"], P.argmax(1))
        row[f"f1_{name}"]  = macro_f1(D["test"]["y"], P)
        row[f"pos_{name}"] = (P.argmax(1) == LABELS.index("positive")).mean()
    row["f1_drop_no_image"] = row["f1_full"] - row["f1_no_image"]
    row["f1_drop_no_text"]  = row["f1_full"] - row["f1_no_text"]
    rob.append(row)
rob = pd.DataFrame(rob).set_index("setup")
print(f"Family: {best_fam}\n" + rob.round(3).to_string())
rob.to_csv(os.path.join(OUT_DIR, "robustness_missing_modality.csv"))

Family: logreg
       acc_full  f1_full  pos_full  acc_no_image  f1_no_image  pos_no_image  acc_no_text  f1_no_text  pos_no_text  f1_drop_no_image  f1_drop_no_text
setup                                                                                                                                               
early     0.718    0.609     0.553         0.653        0.493         0.653        0.698       0.590        0.613             0.116            0.020
late      0.696    0.603     0.523         0.691        0.565         0.626        0.711       0.610        0.606             0.038           -0.008
gated     0.676    0.602     0.481         0.682        0.542         0.606        0.671       0.586        0.528             0.060            0.016


## B8. What the gate learned
The gate is a standardised logistic regression, so each coefficient means: *one standard deviation more of this
input changes the log-odds of the class by this much*. Look at the rows `js_divergence` and `clip_cosine` in particular:
they show how disagreement moves predictions, for example towards neutral, or towards trusting the text.

In [12]:
gate = GATES[(best_fam, 0)]
coef = pd.DataFrame(gate[-1].coef_.T, index=GATE_COLS, columns=gate[-1].classes_)
coef.columns = [LABELS[c] for c in coef.columns]
print(f"Gate coefficients ({best_fam}, seed 0):")
print(coef.round(3).to_string())
coef.to_csv(os.path.join(OUT_DIR, "gate_coefficients.csv"))

# Rows are the true classes and columns the predictions, so off-diagonal cells show which classes get confused
cm = confusion_matrix(yte, PRED[(best_fam, best_setup, 0)], labels=[0, 1, 2])
print(f"\nTest confusion matrix, {best_fam} + {best_setup} (rows = true, cols = predicted):")
print(pd.DataFrame(cm, index=LABELS, columns=LABELS).to_string())

json.dump({"chosen_family": best_fam, "chosen_setup": best_setup, "seeds": SEEDS, "config": CFG[best_fam]},
          open(os.path.join(OUT_DIR, "chosen_model.json"), "w"), indent=1)
print("\nSaved to", OUT_DIR, ":", sorted(os.listdir(OUT_DIR)))

Gate coefficients (logreg, seed 0):
                  negative  neutral  positive
p_text_negative      0.347   -0.100    -0.247
p_text_neutral      -0.114    0.221    -0.107
p_text_positive     -0.240   -0.073     0.314
p_image_negative     0.309   -0.067    -0.242
p_image_neutral     -0.078    0.318    -0.240
p_image_positive    -0.220   -0.186     0.406
clip_cosine         -0.038    0.106    -0.068
js_divergence       -0.022    0.007     0.015
conf_text           -0.082    0.028     0.054
conf_image           0.026   -0.104     0.078

Test confusion matrix, logreg + late (rows = true, cols = predicted):
          negative  neutral  positive
negative        94       14        26
neutral         10       19        10
positive        40       36       198

Saved to /kaggle/working/models : ['all_runs.csv', 'chosen_model.json', 'gate_coefficients.csv', 'per_class_metrics.csv', 'robustness_missing_modality.csv', 'summary_mean_over_seeds.csv', 'tuned_hyperparameters.json']
